# Shuttle 3 Diffusion on a free Colab T4

Jakosc zblizona do Midjourney. Pochodna FLUX.1-schnell, Apache 2.0,
bez bramki HuggingFace - oficjalne repo FLUX wymaga tokena, to nie.
Kwantyzacja 4-bit NF4 miesci 12B parametrow w 16 GB VRAM.

**Runtime -> Change runtime type -> T4 GPU.**

- Komorka 1: instalacja (~3 min)
- Komorka 2: model + interfejs (~6 min przy pierwszym pobraniu, ~12 GB)
- Obraz 1024x1024: ~30-60 s

FLUX rozumie zdania, nie listy tagow. Pisz opisowo:
`a weathered fisherman mending nets at dawn, cold blue light, shallow depth of field`
zamiast `fisherman, nets, dawn, blue, bokeh, 8k, masterpiece`.

In [ ]:
!pip install -q -U diffusers transformers accelerate bitsandbytes sentencepiece protobuf gradio
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, uuid, gc, torch
import gradio as gr
from diffusers import FluxPipeline, FluxTransformer2DModel
from diffusers import BitsAndBytesConfig as DiffusersBnB
from transformers import T5EncoderModel
from transformers import BitsAndBytesConfig as TransformersBnB

# Shuttle 3 Diffusion: pochodna FLUX.1-schnell, Apache 2.0, BEZ bramki HuggingFace.
# Oficjalne black-forest-labs/FLUX.1-* ma gated=auto - akceptacja jest natychmiastowa,
# ale token i tak jest wymagany, wiec domyslnie idziemy bez niego.
REPO = "shuttleai/shuttle-3-diffusion"

# Wariant z oficjalnym FLUX (nieco lepsza estetyka, wymaga tokena):
#   1. Zaakceptuj licencje: https://huggingface.co/black-forest-labs/FLUX.1-schnell
#   2. Wygeneruj token typu Read: https://huggingface.co/settings/tokens
#   3. Colab -> ikona klucza w lewym panelu -> Add new secret -> nazwa: HF_TOKEN
#   4. Odkomentuj cztery linie ponizej:
# from google.colab import userdata
# from huggingface_hub import login
# login(userdata.get("HF_TOKEN"))
# REPO = "black-forest-labs/FLUX.1-schnell"
os.makedirs("out", exist_ok=True)

# Transformer 12B -> 4-bit NF4. Bez tego nie wejdzie w 16 GB.
transformer = FluxTransformer2DModel.from_pretrained(
    REPO, subfolder="transformer", torch_dtype=torch.float16,
    quantization_config=DiffusersBnB(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    ),
)

# Enkoder tekstu T5-XXL tez jest duzy - kwantyzujemy osobno.
text_encoder_2 = T5EncoderModel.from_pretrained(
    REPO, subfolder="text_encoder_2", torch_dtype=torch.float16,
    quantization_config=TransformersBnB(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    ),
)

pipe = FluxPipeline.from_pretrained(
    REPO, transformer=transformer, text_encoder_2=text_encoder_2,
    torch_dtype=torch.float16,
)
pipe.enable_model_cpu_offload()   # T4 ma 16 GB - offload trzyma zapas

RATIOS = {
    "1:1  1024x1024": (1024, 1024),
    "16:9 1344x768": (1344, 768),
    "9:16 768x1344": (768, 1344),
    "3:2  1216x832": (1216, 832),
    "2:3  832x1216": (832, 1216),
}


def generate(prompt, ratio, steps, seed, count):
    if not prompt.strip():
        raise gr.Error("Prompt jest pusty.")

    w, h = RATIOS[ratio]
    seed, count = int(seed), int(count)
    files = []

    for i in range(count):
        s = torch.seed() % (2**31) if seed < 0 else seed + i
        img = pipe(
            prompt=prompt,
            num_inference_steps=int(steps),
            guidance_scale=0.0,          # schnell jest destylowany: guidance musi byc 0
            max_sequence_length=256,     # limit schnell, nie 512
            height=h, width=w,
            generator=torch.Generator("cpu").manual_seed(s),
        ).images[0]

        path = f"out/{uuid.uuid4().hex[:8]}_seed{s}.png"
        img.save(path)
        files.append(path)

        gc.collect()
        torch.cuda.empty_cache()

    seeds = ", ".join(f.split("_seed")[1][:-4] for f in files)
    return files, f"{w}x{h}, {steps} krokow. Seedy: `{seeds}`"


with gr.Blocks(title="FLUX.1-schnell") as demo:
    gr.Markdown("## FLUX.1-schnell &nbsp;·&nbsp; 4-bit NF4 &nbsp;·&nbsp; free Colab T4")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(
                label="Prompt (pelne zdania, nie tagi)", lines=5,
                value=("a weathered fisherman mending nets on a stone pier at dawn, "
                       "cold blue light, sea mist, shallow depth of field, 35mm photograph"),
            )
            ratio = gr.Dropdown(list(RATIOS), value="1:1  1024x1024", label="Proporcje")
            steps = gr.Slider(1, 8, 4, step=1, label="Kroki (schnell: 4 to optimum)")
            seed = gr.Number(-1, label="Seed (-1 = losowy)", precision=0)
            count = gr.Slider(1, 4, 1, step=1, label="Ile obrazow")
            btn = gr.Button("Generuj", variant="primary")
        with gr.Column(scale=2):
            gallery = gr.Gallery(label="Wyniki", columns=2, height=620,
                                 show_download_button=True)
            info = gr.Markdown()

    btn.click(generate, [prompt, ratio, steps, seed, count], [gallery, info])

demo.launch(share=True)